# WSJ 2027 - Utvärdera manuell indelning

Tar de manuellt redigerade Excel-filerna i `input/` (där folk har flyttats av
skäl vi inte hade i datat) och kör samma kvalitetsmått som vår auto-indelning.

**Input:**
- `input/Rundresa Avdelningar.xlsx` (MASTER-bladet)
- `input/Direktresa Avdelningar.xlsx` (MASTER-bladet)

**Output:**
- `output/wsj_<travel>_grupper_manuell.html` per kategori
- Konsol: metrics-tabell + diff mot senaste auto-indelning

Avdelning 99 i Excel-filerna behandlas som "ej placerade" och exkluderas från
metrics så att en handfull personer inte snedvrider snittet.

In [1]:
import sys, importlib
sys.path.insert(0, '/config/notebooks/wsj27')
import wsj27_utils as u
importlib.reload(u)

import pandas as pd
import numpy as np
from collections import Counter

INPUT_DIR = '/config/notebooks/wsj27/input'
OUTPUT_DIR = '/config/notebooks/wsj27/output'
GROUP_SIZE = 36
MAX_KAR = 6
PARKED_GROUP = 99  # Avd 99 = ej placerade, exkluderas från metrics

EXCEL_FILES = {
    'rundresa':   f'{INPUT_DIR}/Rundresa Avdelningar.xlsx',
    'direktresa': f'{INPUT_DIR}/Direktresa Avdelningar.xlsx',
}
AUTO_CSVS = {
    'rundresa':   f'{OUTPUT_DIR}/wsj27_rundresa_grupper.csv',
    'direktresa': f'{OUTPUT_DIR}/wsj27_direktresa_grupper.csv',
}

In [2]:
# Hämta API-data en gång - vi behöver friend_X_name + parent_org till HTML-rapporten,
# och lat/lng till direktresa (Excel-filen saknar dem för den kategorin).
raw = u.fetch_participants()
df_all, _ = u.build_participant_dataframe(raw, include_reserves=True)
print(f'Hämtade {len(df_all)} deltagare totalt')
print(df_all.groupby(['travel', 'category']).size())

Fetched 2703 participants
Confirmed: 2615, Unconfirmed: 6, Cancelled: 82
Loaded 314 samverkansorganisation entries from World Scout Jamboree - Polen 2027, Deltagare _ IST (Funktionar) - deltakere (2).xlsx
Total participants: 2614 (2565 confirmed, 49 reservlista)
Skipped: 6 unconfirmed, 0 nekad (reservlista included), 1 wrong age/no DOB

By category:
category
deltagare    1925
ist           650
cmt            39

By travel type:
travel
rundresa      1501
direktresa     568
egen_resa      298
other          247

Friend wishes:
  With friend 1 member no: 1356
  With friend 2 member no: 805
  With friend 1 name (text): 118
  With friend 2 name (text): 80

Skipped participants:
  DELTAGARE wrong age: Adam Egelnor born 2009-05-25 (age 18)
Hämtade 2614 deltagare totalt
travel      category 
direktresa  deltagare     568
egen_resa   ist           298
other       cmt            39
            ist           208
rundresa    deltagare    1357
            ist           144
dtype: int64


In [3]:
def evaluate_manual(travel, group_size=GROUP_SIZE, max_kar=MAX_KAR):
    """Läs Excel + auto-CSV för en kategori, kör metrics + HTML, returnera (df, total_groups).

    NOTE: Excel-filerna använder globala avdelningsnummer (direktresa 1-16,
    rundresa 17-53). Vi normaliserar internt till 0..N-1 för print_group_metrics
    och remappar för diff-jämförelsen mot auto-CSV (som är 1-indexerad lokal).
    """
    excel_path = EXCEL_FILES[travel]
    auto_csv   = AUTO_CSVS[travel]
    print(f'\n{"="*78}\n{travel.upper()}  ({excel_path})\n{"="*78}')

    # 1. Excel MASTER → member_no → manual group (global numrering)
    xl = pd.read_excel(excel_path, sheet_name='MASTER')
    xl['member_no'] = xl['member_no'].astype(int)
    manual_group = dict(zip(xl['member_no'], xl['group'].astype(int)))
    print(f'Excel MASTER: {len(xl)} deltagare, grupp {xl["group"].min()}..{xl["group"].max()} '
          f'(varav {(xl["group"] == PARKED_GROUP).sum()} i Avd {PARKED_GROUP})')

    # 2. Bygg df utifrån API-datan
    df = (df_all[(df_all['travel'] == travel) & (df_all['category'] == 'deltagare')]
          .copy().reset_index(drop=True))
    df['member_no'] = df['member_no'].astype(int)
    u.assign_coordinates(df, coord_source='kar')
    u.resolve_friend_wishes(df, df_all)
    u.apply_manual_overrides(df, df_all)

    # 3. Filtrera till de som finns i Excel + sätt grupp (global)
    df['group_excel'] = df['member_no'].map(manual_group)
    missing_in_excel = df[df['group_excel'].isna()]
    extra_in_excel = set(xl['member_no']) - set(df['member_no'])
    if len(missing_in_excel):
        print(f'  ⚠ {len(missing_in_excel)} deltagare från API saknas i Excel '
              f'(troligen avhopp efter export)')
    if extra_in_excel:
        print(f'  ⚠ {len(extra_in_excel)} i Excel saknas i API (radera ur sheet?)')
    df = df.dropna(subset=['group_excel']).copy()
    df['group'] = df['group_excel'].astype(int)

    # 4. Plocka bort Avd 99 från metrics
    parked = df[df['group'] == PARKED_GROUP]
    if len(parked):
        print(f'  Plockar bort {len(parked)} från Avd {PARKED_GROUP} ur metrics: '
              + ', '.join(parked['name'].tolist()[:10])
              + (' ...' if len(parked) > 10 else ''))
    df = df[df['group'] != PARKED_GROUP].copy().reset_index(drop=True)

    # 5. Normalisera global → 0-indexerad lokal för print_group_metrics
    sorted_groups = sorted(df['group'].unique())
    remap = {g: i for i, g in enumerate(sorted_groups)}    # {17:0, 18:1, ...} för rundresa
    df['group'] = df['group'].map(remap)
    total_groups = len(sorted_groups)
    group_of = df['group'].values

    # 6. Metrics-tabell (visar lokal 1..N, men varje "Grupp N" motsvarar global N+offset)
    print(f'\n--- Metrics ({total_groups} avdelningar, {len(df)} deltagare) ---')
    print(f'    (Grupp 1 i tabellen = global Avd {sorted_groups[0]}, '
          f'Grupp {total_groups} = global Avd {sorted_groups[-1]})')
    u.print_group_metrics(df, group_of, total_groups, group_size=group_size)

    # 7. HTML-rapport
    friend_wishes = u.build_friend_graph(df)
    report_path = f'{OUTPUT_DIR}/wsj_{travel}_grupper_manuell.html'
    u.generate_groups_report_html(df, total_groups, report_path,
                                  title=f'WSJ 2027 {travel.title()} - Manuell indelning '
                                        f'(Avd {sorted_groups[0]}-{sorted_groups[-1]})',
                                  friend_wishes=friend_wishes,
                                  group_size=group_size,
                                  max_kar=max_kar,
                                  quality='manual',
                                  weight_profile='-',
                                  coord_source='kar')
    print(f'HTML-rapport: {report_path}')

    # 8. Diff mot auto-CSV (auto är 1-indexerad lokal; mappa manual till samma rymd)
    try:
        auto = pd.read_csv(auto_csv)
        auto['member_no'] = auto['member_no'].astype(int)
        auto_group_map = dict(zip(auto['member_no'], auto['group'].astype(int)))
        diffs = []
        n_compared = 0
        for _, row in df.iterrows():
            mno = int(row['member_no'])
            auto_g = auto_group_map.get(mno)
            if auto_g is None:
                continue
            n_compared += 1
            manual_global = manual_group[mno]
            manual_local_1based = remap[manual_global] + 1     # 1-indexerad lokal
            if manual_local_1based != auto_g:
                diffs.append((mno, row['name'], row['kar'], auto_g,
                              manual_local_1based, manual_global))
        print(f'\n--- Diff mot auto-indelning ({auto_csv.split("/")[-1]}) ---')
        print(f'Totalt flyttade: {len(diffs)} av {n_compared} jämförbara '
              f'({100*len(diffs)/max(n_compared,1):.1f}%)')
        if diffs:
            print(f'Första 20 flyttade (lokal-nr → lokal-nr, global-nr inom parentes):')
            for mno, name, kar, fr, to_local, to_global in diffs[:20]:
                print(f'  {mno} {name[:28]:28s} {(kar or "")[:24]:24s} '
                      f'{fr:>3} → {to_local:>3}  (global Avd {to_global})')
            churn = Counter()
            for _, _, _, fr, to_local, _ in diffs:
                churn[fr] += 1
                churn[to_local] += 1
            print(f'\nLokal-grupper med mest aktivitet (in+ut):')
            for g, n in churn.most_common(10):
                global_g = sorted_groups[g-1]
                print(f'  Avd {g:>3} (global {global_g}): {n} ändringar')
    except FileNotFoundError:
        print(f'(Auto-CSV {auto_csv} saknas - hoppar diff)')

    return df, total_groups

## Rundresa

In [4]:
df_run, n_run = evaluate_manual('rundresa')


RUNDRESA  (/config/notebooks/wsj27/input/Rundresa Avdelningar.xlsx)
Excel MASTER: 1330 deltagare, grupp 17..53 (varav 0 i Avd 99)
  (rejected 16 home-address geocodes outside Sweden bbox; will use kår fallback)
Coords by source: home=0, kår=1355, manual=0
Without coordinates (Sweden centroid):
  Alice Jurensons (3348320) — no kår
  Emma Trehn (3357376) — no kår
With coordinates: 1355
Without coordinates (Sweden centroid): 2
=== Text Friend Name Matching ===
Text-only wishes: 111
Matched & applied: 59
Generic wishes (not a person): 4
Unresolved (friend not in this grouping): 48

Matched (verify these are correct):
  Adelia Vallin: "Leo Norstedt. Sancta Maria scoutkår" -> Leo Norstedt (S:ta Maria Scoutkår) [rundresa] via fuzzy(0.96)
  Albin Åkesson: "Malte Ekström Lund" -> Malte Ekström (Lomma scoutkår) [rundresa] via fuzzy(0.84)
  Alex Liljerot Priftis: "Vera Tolvé" -> Vera Tollwé (Årsta Scoutkår) [rundresa] via fuzzy(0.86)
  Alice Jurensons: "Hedvig Seth Torslanda sjöscoutkår" -> Hedv

## Direktresa

In [5]:
df_dir, n_dir = evaluate_manual('direktresa')


DIREKTRESA  (/config/notebooks/wsj27/input/Direktresa Avdelningar.xlsx)
Excel MASTER: 569 deltagare, grupp 1..16 (varav 0 i Avd 99)
  (rejected 16 home-address geocodes outside Sweden bbox; will use kår fallback)
Coords by source: home=0, kår=568, manual=0
With coordinates: 568
Without coordinates (Sweden centroid): 0
=== Text Friend Name Matching ===
Text-only wishes: 45
Matched & applied: 22
Generic wishes (not a person): 0
Unresolved (friend not in this grouping): 23

Matched (verify these are correct):
  Alma Hagström: "Vera Thuresson, Gislaveds" -> Vera Turesson (Equmenia Scout) [direktresa] via fuzzy(0.96)
  Alma Hagström: "Vilda Englund, Gislaved" -> Vilda Englund (Equmenia Scout) [direktresa] via exact
  Alma Oliver Elgh: "Elsa Mattsson Vendelsö Scoutkår" -> Elsa Mattsson (Vendelsö Scoutkår) [direktresa] via exact
  Alma Oliver Elgh: "Elsa Matssson Vendelsö scoutkår" -> Elsa Mattsson (Vendelsö Scoutkår) [direktresa] via fuzzy(0.92)+kar
  Arvid Bergsten: "Samuel Nordström, Toll

## Skjut avdelningar tillbaka till Scoutnet

Tar `member_no` → `group`-mappningen från Excel MASTER-bladen och postar dem
till `/api/project/checkin`. Avdelning 99 (ej placerade) hoppas över.

**Numrering globalt över kategorier (redan applicerad i Excel-filerna):**
- **Direktresa**: avdelning **1-16**
- **Rundresa**: avdelning **17-53**
- **Avd 99**: ej placerade (hoppas över)

Cellen kontrollerar att Excel-spannen matchar förväntningen innan dry-run körs.

**Säkerhet:**
- `dry_run=True` (default) skickar inget — visar bara vad som hade skickats
- `status` skickas inte → `checked_in`/`attended` påverkas aldrig
- Verifierat på Ida Sandholdt att endpoint:en är idempotent

Sätt `dry_run=False` när du är säker.

In [7]:
# Reload utils så ändringar i wsj27_utils.py syns utan kernel-restart
import importlib
importlib.reload(u)

# Förväntade avdelningsspann per kategori
EXPECTED_RANGES = {
    'direktresa': (1, 16),
    'rundresa':   (17, 53),
}


def build_member_to_avdelning_from_excel(excel_path):
    """Läs MASTER och returnera {member_no: avdelning} - värdena används direkt."""
    xl = pd.read_excel(excel_path, sheet_name='MASTER')
    return dict(zip(xl['member_no'].astype(int), xl['group'].astype(int)))


def sanity_check_range(travel, mapping):
    lo_expected, hi_expected = EXPECTED_RANGES[travel]
    non_parked = [v for v in mapping.values() if v != PARKED_GROUP]
    lo, hi = min(non_parked), max(non_parked)
    n_unique = len(set(non_parked))
    n_parked = sum(1 for v in mapping.values() if v == PARKED_GROUP)
    ok = (lo == lo_expected and hi == hi_expected)
    print(f'  Spann: {lo}..{hi} ({n_unique} unika avd, {n_parked} i Avd {PARKED_GROUP})')
    if not ok:
        print(f'  ⚠ FÖRVÄNTAT {lo_expected}..{hi_expected} - kolla Excel-filen!')
    return ok


# Dry run för bägge kategorier
for travel, path in EXCEL_FILES.items():
    print(f'\n--- {travel.upper()} ---')
    mapping = build_member_to_avdelning_from_excel(path)
    sanity_check_range(travel, mapping)
    u.push_avdelningar_to_scoutnet(mapping, skip_groups=(PARKED_GROUP,), dry_run=True)

# För att verkligen skicka (kör en kategori i taget):
u.push_avdelningar_to_scoutnet(
    build_member_to_avdelning_from_excel(EXCEL_FILES['direktresa']),
    skip_groups=(PARKED_GROUP,), dry_run=False,
)
u.push_avdelningar_to_scoutnet(
    build_member_to_avdelning_from_excel(EXCEL_FILES['rundresa']),
    skip_groups=(PARKED_GROUP,), dry_run=False,
)


--- RUNDRESA ---
  Spann: 17..53 (37 unika avd, 0 i Avd 99)
Total att skicka: 1330 (hoppade över 0)

DRY RUN - inget skickas. Första 3 i payload:
  3351398: {'questions': {'88168': {'value': '17'}}}
  3379410: {'questions': {'88168': {'value': '17'}}}
  3377116: {'questions': {'88168': {'value': '17'}}}

--- DIREKTRESA ---
  Spann: 1..16 (16 unika avd, 0 i Avd 99)
Total att skicka: 569 (hoppade över 0)

DRY RUN - inget skickas. Första 3 i payload:
  3428576: {'questions': {'88168': {'value': '1'}}}
  3396220: {'questions': {'88168': {'value': '1'}}}
  3351167: {'questions': {'88168': {'value': '1'}}}
Total att skicka: 569 (hoppade över 0)
  Chunk 1: updated=0 unchanged=200 not_found=0 no_member=0
  Chunk 2: updated=0 unchanged=400 not_found=0 no_member=1
  Chunk 3: updated=0 unchanged=569 not_found=0 no_member=1

=== Totalt ===
  Uppdaterade (faktiskt ändrade): 0
  Oförändrade (redan rätt värde): 569
  Hittades inte:                  0
  Saknar medlemskap:              1
  Överhoppade

{'updated': 1,
 'unchanged': 1329,
 'not_found': 0,
 'no_member': 0,
 'skipped': 0,
 'errors': []}